In [10]:
import requests
import pandas as pd
from datetime import datetime, timedelta, date
import hashlib
import io

# from airflow import DAG
# from airflow.operators.python import PythonOperator
# from airflow.providers.amazon.aws.hooks.s3 import S3Hook


pd.set_option('display.max_columns', 300)
pd.set_option('display.max_rows', 15) #test
pd.set_option('display.float_format', '{:.2f}'.format)

# Документация
# https://partner.steamgames.com/doc/store/getreviews

Мы собираем только новые отзывы по timestamp_created, без поддержки последующих edits(когда появляются обновленные отзывы, то старые не перезаписываем).

app_id=730 - CS 2
Одна строка = один review.
Уникальный ключ - recommendationid
Бизнес-дата - review_created_date = date(timestamp_created) отзыв навсегда относится к дню своего создания

Для конкретной даты target_date:
идёт в Steam Reviews API
читает ленту через filter=recent
обходит страницы через cursor
из всех полученных reviews оставляет только те, у которых date(timestamp_created) == target_date
дедуплицирует по recommendationid
пишет один parquet за день
кладёт его в MinIO по фиксированному пути
при повторном запуске за ту же дату просто перезаписывает этот же объект

Api клиент - GET/JSON
Пагинация - cursor (переход по страницам, остановка цикла), сырые reviews
Бизнес фильтр - перевод timestamp в date, отбор строк за target_date, нормализацию структуры, дедуп
Сбор DF, экспорт и загрузка в паркет
Орекстрация - вызывает остальное, app_id, target_date

для пары (app_id, target_date) всегда один и тот же output path
И при перезапуске:
собираем данные заново
перезаписываем parquet по тому же пути

Пагинация:
Так как filter=recent даёт ленту по времени создания, то для target_date:
пока на странице встречаются даты >= target_date, продолжаем 
когда вся страница уже < target_date, прекращаем
То есть stop condition опирается на timestamp_created.

Список функций:
1) fetch_reviews_page(...) Пуляет GET-запрос
2) меняем параметр курсор для пагинации.
3) 


In [4]:

BASE_URL  = "https://store.steampowered.com/appreviews/"
REQUEST_TIMEOUT = 30
MAX_RETRIES = 5
RETRY_SLEEP_SECONDS = 2



In [5]:
def fetch_reviews_page(app_id, params):
    """
    Один GET-запрос к Steam Reviews API.
    Возвращает сырой JSON payload.
    """
    url = f"https://store.steampowered.com/appreviews/{app_id}"

    response = requests.get(url, params=params, timeout=30)
    payload = response.json()

    if payload.get("success") != 1:
        raise ValueError(f"Steam API вернул залупу, а не success={payload.get('success')}")

    return payload

In [6]:
# def export_reviews_to_csv(df, app_id, target_date, output_dir="data_test"):
    
#     import os
    
#     os.makedirs(output_dir, exist_ok=True)

#     file_name = f"steam_reviews_app_{app_id}_{target_date}.csv"
#     file_path = os.path.join(output_dir, file_name)

#     df.to_csv(file_path, index=False, encoding="utf-8-sig")

#     print(f"CSV сохранён: {file_path}")

#     return file_path


In [7]:
params = {
    "json": 1,
    "language": "all",
    "filter": "recent",
    "num_per_page": 100,
    "cursor": "*"
}

In [8]:
def collect_reviews_for_date(app_id, target_date, params):
    params = params.copy()
    all_pages = [] # Список под датафреймы с отзывами

    while True:
        payload = fetch_reviews_page(app_id=app_id, params=params)
        reviews = payload["reviews"]

        if len(reviews) == 0:
            print("Отзывы закончились, остановка")
            break

        page_df = pd.DataFrame(reviews)

        author_df = pd.json_normalize(page_df["author"])
        author_df.columns = [f"author_{col}" for col in author_df.columns]
        page_df = pd.concat([page_df.drop(columns=["author"]), author_df], axis=1)

        page_df["timestamp_created_dt"] = pd.to_datetime(
            page_df["timestamp_created"], unit="s", utc=True
        )
        page_df["created_date"] = page_df["timestamp_created_dt"].dt.date

        page_df["timestamp_updated_dt"] = pd.to_datetime(
            page_df["timestamp_updated"], unit="s", utc=True
        )
        page_df["updated_date"] = page_df["timestamp_updated_dt"].dt.date

        filtered_page_df = page_df[page_df["created_date"] == target_date].copy() # Фильтруем df по дате

        all_pages.append(filtered_page_df)

        print(
            f"Итерация: всего строк={len(page_df)}, "
            f"за нужную дату={len(filtered_page_df)}, "
            f"min_date={page_df['created_date'].min()}, "
            f"max_date={page_df['created_date'].max()}"
        )

        if (page_df["created_date"] < target_date).all(): # Стопаем если все страница старше target_date
            print(f"Текущая страница уже старше {target_date}, остановка")
            break

        params["cursor"] = payload["cursor"] # Для пагинации обновляем параметр после каждого запроса GET

    if len(all_pages) == 0:
        return pd.DataFrame()

    result_df = pd.concat(all_pages, ignore_index=True)

    return result_df


In [11]:
df_test_reviews = collect_reviews_for_date(
    app_id=730,
    target_date=date(2026, 3, 10),
    params=params
)

Итерация: всего строк=100, за нужную дату=0, min_date=2026-03-15, max_date=2026-03-15
Итерация: всего строк=100, за нужную дату=0, min_date=2026-03-15, max_date=2026-03-15
Итерация: всего строк=100, за нужную дату=0, min_date=2026-03-15, max_date=2026-03-15
Итерация: всего строк=100, за нужную дату=0, min_date=2026-03-15, max_date=2026-03-15
Итерация: всего строк=100, за нужную дату=0, min_date=2026-03-15, max_date=2026-03-15
Итерация: всего строк=100, за нужную дату=0, min_date=2026-03-15, max_date=2026-03-15
Итерация: всего строк=100, за нужную дату=0, min_date=2026-03-14, max_date=2026-03-15
Итерация: всего строк=100, за нужную дату=0, min_date=2026-03-14, max_date=2026-03-14
Итерация: всего строк=100, за нужную дату=0, min_date=2026-03-14, max_date=2026-03-14
Итерация: всего строк=100, за нужную дату=0, min_date=2026-03-14, max_date=2026-03-14
Итерация: всего строк=100, за нужную дату=0, min_date=2026-03-14, max_date=2026-03-14
Итерация: всего строк=100, за нужную дату=0, min_date=

In [ ]:
# v2
def collect_reviews_for_date(app_id, target_date, params):
    params = params.copy()
    all_pages = []  # Список под датафреймы с отзывами

    while True:
        payload = fetch_reviews_page(app_id=app_id, params=params)
        reviews = payload.get("reviews", [])

        # ИЗМЕНЕНИЕ: этот break оставляем обязательно, иначе цикл может уйти в бесконечность
        # или упасть на попытке обработать пустую страницу
        if len(reviews) == 0:
            print("Отзывы закончились, остановка")
            break

        page_df = pd.DataFrame(reviews)

        author_df = pd.json_normalize(page_df["author"])
        author_df.columns = [f"author_{col}" for col in author_df.columns]
        page_df = pd.concat([page_df.drop(columns=["author"]), author_df], axis=1)

        page_df["timestamp_created_dt"] = pd.to_datetime(
            page_df["timestamp_created"], unit="s", utc=True
        )
        page_df["created_date"] = page_df["timestamp_created_dt"].dt.date

        page_df["timestamp_updated_dt"] = pd.to_datetime(
            page_df["timestamp_updated"], unit="s", utc=True
        )
        page_df["updated_date"] = page_df["timestamp_updated_dt"].dt.date

        filtered_page_df = page_df[page_df["created_date"] == target_date].copy()  # Фильтруем df по дате

        # ИЗМЕНЕНИЕ: оставляем как ты хочешь — даже пустые df складываем в список
        all_pages.append(filtered_page_df)

        print(
            f"Итерация: всего строк={len(page_df)}, "
            f"за нужную дату={len(filtered_page_df)}, "
            f"min_date={page_df['created_date'].min()}, "
            f"max_date={page_df['created_date'].max()}"
        )

        if (page_df["created_date"] < target_date).all():  # Стопаем если вся страница старше target_date
            print(f"Текущая страница уже старше {target_date}, остановка")
            break

        params["cursor"] = payload["cursor"]  # Для пагинации обновляем параметр после каждого запроса GET

    if len(all_pages) == 0:
        return pd.DataFrame()

    result_df = pd.concat(all_pages, ignore_index=True)

    return result_df


def run_reviews_job(target_date, app_id=730):
    """
    Основной job:
    - сначала проверяет, есть ли уже parquet за target_date в MinIO
    - если есть, пропускает загрузку
    - если нет, собирает отзывы за дату
    - пишет parquet в память
    - грузит в MinIO через S3Hook
    """

    target_date = datetime.strptime(target_date, "%Y-%m-%d").date()

    params = {
        "json": 1,
        "language": "all",
        "filter": "recent",
        "num_per_page": 100,
        "cursor": "*"
    }

    hook = S3Hook(aws_conn_id="minios3_conn")

    # ИЗМЕНЕНИЕ: убрали hash, сделали читаемый key
    key = f"EvgSergeev_steam_reviews/app_id={app_id}/dt={target_date}/reviews.parquet"

    # ИЗМЕНЕНИЕ: сначала проверяем наличие файла в MinIO, и только потом идем в Steam API
    if hook.check_for_key(key=key, bucket_name="dev"):
        print(f"Файл за дату {target_date} уже существует в MinIO, загрузку пропускаем")
        return {
            "status": "skipped_already_exists",
            "app_id": app_id,
            "target_date": str(target_date),
            "bucket": "dev",
            "key": key
        }

    df = collect_reviews_for_date(
        app_id=app_id,
        target_date=target_date,
        params=params
    )

    # ИЗМЕНЕНИЕ: блок no_data убрали — теперь даже пустой df пишем в parquet
    buffer = io.BytesIO()
    df.to_parquet(buffer, index=False)
    buffer.seek(0)

    hook.load_bytes(
        bytes_data=buffer.read(),
        key=key,
        bucket_name="dev",
        replace=True
    )

    print(f"Файл успешно загружен в s3://dev/{key}")

    return {
        "status": "uploaded",
        "app_id": app_id,
        "target_date": str(target_date),
        "rows": int(len(df)),
        "bucket": "dev",
        "key": key
    }

In [ ]:
# v3
import io
import requests
import pandas as pd

from datetime import datetime, timedelta

from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.providers.amazon.aws.hooks.s3 import S3Hook


def fetch_reviews_page(app_id, params):
    """
    Один GET-запрос к Steam Reviews API.
    Возвращает сырой JSON payload.
    """
    url = f"https://store.steampowered.com/appreviews/{app_id}"

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    payload = response.json()

    if payload.get("success") != 1:
        raise ValueError(f"Steam API вернул не success=1, а success={payload.get('success')}")

    return payload


def collect_reviews_for_date(app_id, target_date, params):
    params = params.copy()
    all_pages = []  # Список под датафреймы с отзывами

    while True:
        payload = fetch_reviews_page(app_id=app_id, params=params)
        reviews = payload.get("reviews", [])

        # Останавливаемся, если API больше не отдает отзывы
        if len(reviews) == 0:
            print("Отзывы закончились, остановка")
            break

        page_df = pd.DataFrame(reviews)

        author_df = pd.json_normalize(page_df["author"])
        author_df.columns = [f"author_{col}" for col in author_df.columns]
        page_df = pd.concat([page_df.drop(columns=["author"]), author_df], axis=1)

        page_df["timestamp_created_dt"] = pd.to_datetime(
            page_df["timestamp_created"], unit="s", utc=True
        )
        page_df["created_date"] = page_df["timestamp_created_dt"].dt.date

        page_df["timestamp_updated_dt"] = pd.to_datetime(
            page_df["timestamp_updated"], unit="s", utc=True
        )
        page_df["updated_date"] = page_df["timestamp_updated_dt"].dt.date

        filtered_page_df = page_df[page_df["created_date"] == target_date].copy()

        # Оставляем как есть: даже пустые куски можно складывать, concat их переживет
        all_pages.append(filtered_page_df)

        print(
            f"Итерация: всего строк={len(page_df)}, "
            f"за нужную дату={len(filtered_page_df)}, "
            f"min_date={page_df['created_date'].min()}, "
            f"max_date={page_df['created_date'].max()}"
        )

        # Если вся страница уже старше target_date, дальше искать бессмысленно
        if (page_df["created_date"] < target_date).all():
            print(f"Текущая страница уже старше {target_date}, остановка")
            break

        params["cursor"] = payload["cursor"]


    if len(all_pages) == 0:
        return pd.DataFrame()

    result_df = pd.concat(all_pages, ignore_index=True)
    return result_df


def run_reviews_job(target_date, app_id=730):
    """
    Основной job:
    1. Проверяем, есть ли parquet за target_date в MinIO
    2. Если есть — пропускаем загрузку
    3. Если нет — собираем отзывы из Steam API
    4. Пишем parquet в память
    5. Грузим parquet в MinIO
    """
    target_date = datetime.strptime(target_date, "%Y-%m-%d").date()

    params = {
        "json": 1,
        "language": "all",
        "filter": "recent",
        "num_per_page": 100,
        "cursor": "*"
    }

    hook = S3Hook(aws_conn_id="minios3_conn")

    key = f"EvgSergeev_steam_reviews/app_id={app_id}/dt={target_date}/reviews.parquet"

    # Проверка идемпотентности: если файл уже есть, второй раз дату не собираем
    if hook.check_for_key(key=key, bucket_name="dev"):
        print(f"Файл за дату {target_date} уже существует в MinIO, загрузку пропускаем")
        return {
            "status": "skipped_already_exists",
            "app_id": app_id,
            "target_date": str(target_date),
            "bucket": "dev",
            "key": key
        }

    df = collect_reviews_for_date(
        app_id=app_id,
        target_date=target_date,
        params=params
    )

    # Даже если df пустой — все равно пишем parquet
    buffer = io.BytesIO()
    df.to_parquet(buffer, index=False)
    buffer.seek(0)

    hook.load_bytes(
        bytes_data=buffer.read(),
        key=key,
        bucket_name="dev",
        replace=True
    )

    print(f"Файл успешно загружен в s3://dev/{key}")

    return {
        "status": "uploaded",
        "app_id": app_id,
        "target_date": str(target_date),
        "rows": int(len(df)),
        "bucket": "dev",
        "key": key
    }


default_args = {
    "owner": "evgeny",
    "start_date": datetime(2026, 3, 1),
    "retries": 2,
    "retry_delay": timedelta(minutes=5),
}

dag = DAG(
    dag_id="steam_reviews_api_to_s3",
    default_args=default_args,
    schedule="0 10 * * *",
    catchup=True,
    description="Steam reviews API to MinIO parquet",
    tags=["steam", "reviews", "api", "s3", "minio"],
)

upload_reviews_to_s3 = PythonOperator(
    task_id="upload_reviews_to_s3",
    python_callable=run_reviews_job,
    op_kwargs={
        "target_date": "{{ ds }}",
        "app_id": 730
    },
    dag=dag,
)

In [42]:
# df = collect_reviews_for_date(
#     app_id=730,
#     target_date=date(2026, 3, 13),
#     params=params
# )

Итерация: всего строк=100, за нужную дату=0, min_date=2026-03-14, max_date=2026-03-14
Итерация: всего строк=100, за нужную дату=0, min_date=2026-03-14, max_date=2026-03-14
Итерация: всего строк=100, за нужную дату=0, min_date=2026-03-14, max_date=2026-03-14
Итерация: всего строк=100, за нужную дату=0, min_date=2026-03-14, max_date=2026-03-14
Итерация: всего строк=100, за нужную дату=0, min_date=2026-03-14, max_date=2026-03-14
Итерация: всего строк=100, за нужную дату=0, min_date=2026-03-14, max_date=2026-03-14
Итерация: всего строк=100, за нужную дату=0, min_date=2026-03-14, max_date=2026-03-14
Итерация: всего строк=99, за нужную дату=0, min_date=2026-03-14, max_date=2026-03-14
Итерация: всего строк=100, за нужную дату=0, min_date=2026-03-14, max_date=2026-03-14
Итерация: всего строк=100, за нужную дату=0, min_date=2026-03-14, max_date=2026-03-14
Итерация: всего строк=100, за нужную дату=0, min_date=2026-03-14, max_date=2026-03-14
Итерация: всего строк=100, за нужную дату=0, min_date=2

In [45]:
# Тестовый экспорт
# export_reviews_to_csv(df, 570, "2026-03-13")

CSV сохранён: data_test/steam_reviews_app_570_2026-03-13.csv


'data_test/steam_reviews_app_570_2026-03-13.csv'

In [ ]:
# target_date = pd.to_datetime(target_date).date()

In [24]:
new_df_test = pd.DataFrame(payload['reviews'])

In [29]:
new_df_test.head(5)

,recommendationid,author,language,review,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,refunded,written_during_early_access,primarily_steam_deck,app_release_date,hardware,reactions
0,220376551,"{'steamid': '76561198013630705', 'personaname'...",russian,весело и идет даже на калькуляторах на ультрах...,1773175358,1773175358,True,0,0,0.50,0,True,False,False,False,False,1373389200,"{'manufacturer': 'Rikor Electronics', 'model':...",[]
1,220286904,"{'steamid': '76561198080145423', 'personaname'...",russian,"игра дерьма,дают мне бан буквально НИ-ЗА-ЧТО",1773075706,1773075706,False,0,0,0.50,0,True,False,False,False,False,1373389200,NaN,[]
2,220181220,"{'steamid': '76561198060864509', 'personaname'...",russian,Топчик,1772974939,1772974939,True,0,0,0.50,0,True,False,False,False,False,1373389200,NaN,[]
3,220113546,"{'steamid': '76561198015648484', 'personaname'...",brazilian,Jogo Toxico,1772912606,1772912606,False,0,1,0.50,0,True,False,False,False,False,1373389200,NaN,[]
4,219814692,"{'steamid': '76561197982572395', 'personaname'...",russian,Ваша жопа сгорит быстрее спички. Но если любит...,1772645537,1772645537,False,0,0,0.50,0,True,False,False,False,False,1373389200,NaN,[]


In [ ]:
volvo_app_id = 730

Endpoint_volvo = f"{API_BASE_VOLVO}{volvo_app_id}" # подробности по игре

volvo_params = {
        "json": 1,
        "language": "all",
        "filter": "recent",
        "num_per_page": 100,
        "cursor": "*"
    }

response_volvo = requests.get(Endpoint_volvo, params=volvo_params)
print(response_volvo)
data_volvo = response_volvo.json()

print(type(data_volvo))
print(data_volvo.keys())

In [3]:
print(data_volvo.keys())

dict_keys(['success', 'query_summary', 'reviews', 'cursor'])


In [4]:
volvo_df = pd.DataFrame(data_volvo['reviews'])

In [ ]:
df['timestamp_created'] = pd.to_datetime(df['timestamp_created'], unit = 's')

In [16]:
volvo_df.head(10)

,recommendationid,author,language,review,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,refunded,written_during_early_access,primarily_steam_deck,app_release_date,reactions,hardware
0,220659972,"{'steamid': '76561199529972100', 'personaname'...",russian,не советую.,2026-03-14 15:26:15,2026-03-14 15:26:15,False,0,0,0.50,0,True,False,False,False,False,2012-08-21 17:00:00,[],NaN
1,220659939,"{'steamid': '76561198747602510', 'personaname'...",russian,cs go global offensive получше,2026-03-14 15:25:53,2026-03-14 15:25:53,False,0,0,0.50,0,True,False,False,False,False,2012-08-21 17:00:00,[],NaN
2,220659929,"{'steamid': '76561199070665650', 'personaname'...",hungarian,Elindul. \n:-),2026-03-14 15:25:44,2026-03-14 15:25:44,True,1,0,0.523809552192687988,0,True,False,False,False,False,2012-08-21 17:00:00,[],NaN
3,220659912,"{'steamid': '76561199762486008', 'personaname'...",schinese,HW,2026-03-14 15:25:26,2026-03-14 15:25:26,True,0,0,0.50,0,True,False,False,False,False,2012-08-21 17:00:00,[],NaN
4,220659870,"{'steamid': '76561198292891696', 'personaname'...",schinese,https://steamcommunity.com/id/jL-13/inventory/...,2026-03-14 15:24:52,2026-03-14 15:24:52,True,0,0,0.50,0,True,False,False,False,False,2012-08-21 17:00:00,[],NaN
5,220659859,"{'steamid': '76561199661787351', 'personaname'...",russian,Не регает вообще,2026-03-14 15:24:45,2026-03-14 15:24:45,True,0,0,0.50,1,True,False,False,False,False,2012-08-21 17:00:00,[],NaN
6,220659856,"{'steamid': '76561198045121326', 'personaname'...",russian,стрилялки\r\n,2026-03-14 15:24:44,2026-03-14 15:24:44,True,0,0,0.50,0,True,False,False,False,False,2012-08-21 17:00:00,[],NaN
7,220659823,"{'steamid': '76561199187537457', 'personaname'...",english,llrud orj ired tiir,2026-03-14 15:24:19,2026-03-14 15:24:19,True,0,0,0.50,0,True,False,False,False,False,2012-08-21 17:00:00,[],NaN
8,220659729,"{'steamid': '76561198727807073', 'personaname'...",english,fmm de joc,2026-03-14 15:23:10,2026-03-14 15:23:10,True,0,0,0.50,0,True,False,False,False,False,2012-08-21 17:00:00,[],NaN
9,220659722,"{'steamid': '76561199479861169', 'personaname'...",ukrainian,"игра имба, но калопоедатели скажут что кс го л...",2026-03-14 15:23:06,2026-03-14 15:23:06,True,0,0,0.50,0,True,False,False,False,False,2012-08-21 17:00:00,[],NaN


In [6]:
author_df = pd.json_normalize(volvo_df["author"])

In [7]:

# df = pd.concat([df.drop(columns=["author"]), author_df], axis=1)
volvo_df["timestamp_created"] = pd.to_datetime(volvo_df["timestamp_created"], unit="s")

In [8]:
volvo_df["timestamp_updated"] = pd.to_datetime(volvo_df["timestamp_updated"], unit="s")
volvo_df["app_release_date"] = pd.to_datetime(volvo_df["app_release_date"], unit="s")

/tmp/ipykernel_342167/3047818422.py:2: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  volvo_df["app_release_date"] = pd.to_datetime(volvo_df["app_release_date"], unit="s")


In [15]:
author_df

,steamid,personaname,persona_status,profile_url,num_games_owned,num_reviews,playtime_forever,playtime_last_two_weeks,playtime_at_review,last_played,avatar,deck_playtime_at_review
0,76561199529972100,alfach,online,https://steamcommunity.com/profiles/7656119952...,0,1,3675,1233,3675,1773501632,164548ad697a62dc482a292f2721763dd9148bdc,NaN
1,76561198747602510,Саня_Sanya,in-game,https://steamcommunity.com/profiles/7656119874...,2,2,1915,330,1915,1773502008,8b439d628bf5059afc8220cb9aa4bc70012df78e,NaN
2,76561199070665650,LIL_Gala,online,https://steamcommunity.com/profiles/7656119907...,0,1,42180,1023,42180,1773424550,148ff422f2245ab66abfeabf3f7506861d6b703b,NaN
3,76561199762486008,东京闪景,in-game,https://steamcommunity.com/profiles/7656119976...,29,1,30015,873,30015,1773501917,2f61dc1e9709a2119b4735226c4a159894d3de42,NaN
4,76561198292891696,海绵宝宝,offline,https://steamcommunity.com/profiles/7656119829...,64,1,5049,331,5049,1773327295,26dcdc1c27b35589d995d76e5a4aa69088b757cf,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
95,76561199686354724,GEMS,offline,https://steamcommunity.com/profiles/7656119968...,5,2,8921,1745,8921,1773499486,483f7747344206f4ebc39cfa318daa9fafcff512,NaN
96,76561199528496305,uzuy666,in-game,https://steamcommunity.com/profiles/7656119952...,56,3,11116,956,11116,1773500932,f9e4e0d0bad4ed15082545bf0ea5cb7178cfc63b,NaN
97,76561199520881135,Chtooo,online,https://steamcommunity.com/profiles/7656119952...,0,2,1456,647,1456,1773498770,148ff422f2245ab66abfeabf3f7506861d6b703b,NaN
98,76561199854848083,DELLROYY,online,https://steamcommunity.com/profiles/7656119985...,0,2,9048,2226,9018,1773501293,815620bf2491be691e9cd2012741e1cba2f0fb06,NaN


In [14]:
volvo_df.head(10)

,recommendationid,author,language,review,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,refunded,written_during_early_access,primarily_steam_deck,app_release_date,reactions,hardware
0,220659972,"{'steamid': '76561199529972100', 'personaname'...",russian,не советую.,2026-03-14 15:26:15,2026-03-14 15:26:15,False,0,0,0.50,0,True,False,False,False,False,2012-08-21 17:00:00,[],NaN
1,220659939,"{'steamid': '76561198747602510', 'personaname'...",russian,cs go global offensive получше,2026-03-14 15:25:53,2026-03-14 15:25:53,False,0,0,0.50,0,True,False,False,False,False,2012-08-21 17:00:00,[],NaN
2,220659929,"{'steamid': '76561199070665650', 'personaname'...",hungarian,Elindul. \n:-),2026-03-14 15:25:44,2026-03-14 15:25:44,True,1,0,0.523809552192687988,0,True,False,False,False,False,2012-08-21 17:00:00,[],NaN
3,220659912,"{'steamid': '76561199762486008', 'personaname'...",schinese,HW,2026-03-14 15:25:26,2026-03-14 15:25:26,True,0,0,0.50,0,True,False,False,False,False,2012-08-21 17:00:00,[],NaN
4,220659870,"{'steamid': '76561198292891696', 'personaname'...",schinese,https://steamcommunity.com/id/jL-13/inventory/...,2026-03-14 15:24:52,2026-03-14 15:24:52,True,0,0,0.50,0,True,False,False,False,False,2012-08-21 17:00:00,[],NaN
5,220659859,"{'steamid': '76561199661787351', 'personaname'...",russian,Не регает вообще,2026-03-14 15:24:45,2026-03-14 15:24:45,True,0,0,0.50,1,True,False,False,False,False,2012-08-21 17:00:00,[],NaN
6,220659856,"{'steamid': '76561198045121326', 'personaname'...",russian,стрилялки\r\n,2026-03-14 15:24:44,2026-03-14 15:24:44,True,0,0,0.50,0,True,False,False,False,False,2012-08-21 17:00:00,[],NaN
7,220659823,"{'steamid': '76561199187537457', 'personaname'...",english,llrud orj ired tiir,2026-03-14 15:24:19,2026-03-14 15:24:19,True,0,0,0.50,0,True,False,False,False,False,2012-08-21 17:00:00,[],NaN
8,220659729,"{'steamid': '76561198727807073', 'personaname'...",english,fmm de joc,2026-03-14 15:23:10,2026-03-14 15:23:10,True,0,0,0.50,0,True,False,False,False,False,2012-08-21 17:00:00,[],NaN
9,220659722,"{'steamid': '76561199479861169', 'personaname'...",ukrainian,"игра имба, но калопоедатели скажут что кс го л...",2026-03-14 15:23:06,2026-03-14 15:23:06,True,0,0,0.50,0,True,False,False,False,False,2012-08-21 17:00:00,[],NaN


In [10]:
author_df

,steamid,personaname,persona_status,profile_url,num_games_owned,num_reviews,playtime_forever,playtime_last_two_weeks,playtime_at_review,last_played,avatar,deck_playtime_at_review
0,76561199529972100,alfach,online,https://steamcommunity.com/profiles/7656119952...,0,1,3675,1233,3675,1773501632,164548ad697a62dc482a292f2721763dd9148bdc,NaN
1,76561198747602510,Саня_Sanya,in-game,https://steamcommunity.com/profiles/7656119874...,2,2,1915,330,1915,1773502008,8b439d628bf5059afc8220cb9aa4bc70012df78e,NaN
2,76561199070665650,LIL_Gala,online,https://steamcommunity.com/profiles/7656119907...,0,1,42180,1023,42180,1773424550,148ff422f2245ab66abfeabf3f7506861d6b703b,NaN
3,76561199762486008,东京闪景,in-game,https://steamcommunity.com/profiles/7656119976...,29,1,30015,873,30015,1773501917,2f61dc1e9709a2119b4735226c4a159894d3de42,NaN
4,76561198292891696,海绵宝宝,offline,https://steamcommunity.com/profiles/7656119829...,64,1,5049,331,5049,1773327295,26dcdc1c27b35589d995d76e5a4aa69088b757cf,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
95,76561199686354724,GEMS,offline,https://steamcommunity.com/profiles/7656119968...,5,2,8921,1745,8921,1773499486,483f7747344206f4ebc39cfa318daa9fafcff512,NaN
96,76561199528496305,uzuy666,in-game,https://steamcommunity.com/profiles/7656119952...,56,3,11116,956,11116,1773500932,f9e4e0d0bad4ed15082545bf0ea5cb7178cfc63b,NaN
97,76561199520881135,Chtooo,online,https://steamcommunity.com/profiles/7656119952...,0,2,1456,647,1456,1773498770,148ff422f2245ab66abfeabf3f7506861d6b703b,NaN
98,76561199854848083,DELLROYY,online,https://steamcommunity.com/profiles/7656119985...,0,2,9048,2226,9018,1773501293,815620bf2491be691e9cd2012741e1cba2f0fb06,NaN


In [12]:
#volvo_df.to_csv('CS_2_reviews_df.csv')

In [13]:
# Ограничения Allowed poll rate - 1 request per second for most requests, 1 request per 60 seconds for the *all* requests.
API_BASE_SPY = "https://steamspy.com/api.php"

spy_pages = [0,1,2,3]

Endpoint_spy = f"{API_BASE_SPY}request=all&page=" # список всех игр

SPY_params = {
    "request": "all",
    "page": 0 }

response_spy = requests.get(API_BASE_SPY, params = SPY_params)
print(response_spy)
data_spy = response_spy.json()

print(type(data_spy))

KeyboardInterrupt: 

In [ ]:
# Тут id игр с нулевой страницы
data_spy.keys()

In [ ]:
df_spy_data = pd.DataFrame([data_spy['730']])


In [ ]:
df_spy_data
